# IMC08 — Assignment: Dense GEMV & Sparse SpMV Benchmarks
**Track A — PCAM Modernization**

**Citation:** Starter structure adapted from IMC08 module 'Code & Scripts Across Assignment Tracks'.  
All benchmark code, modifications, and analysis are my own work.

---
## Notebook Overview
This notebook benchmarks Matrix-Vector Multiplication (y = Ax) across:
1. **CPU (NumPy)** — dense GEMV
2. **GPU (CuPy / cuBLAS)** — dense GEMV
3. **CPU Sparse (SciPy CSR)** — SpMV
4. **GPU Sparse (cuSPARSE via CuPy)** — SpMV

All operations use **float64 (FP64)**. Results reported in: runtime (ms), GFLOP/s, and effective GB/s.

## Cell 1 — Runtime Check & Device Info

In [ ]:
# ── Cell 1: Device Info ──────────────────────────────────────────────────────
# My own code — prints CPU and GPU hardware info for reproducibility

import subprocess, platform, multiprocessing

print("=" * 60)
print("DEVICE INFORMATION")
print("=" * 60)

# CPU info
print(f"Python version : {platform.python_version()}")
print(f"OS             : {platform.system()} {platform.release()}")
print(f"CPU cores      : {multiprocessing.cpu_count()}")

# GPU info via nvidia-smi
try:
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version,compute_cap",
         "--format=csv,noheader"],
        capture_output=True, text=True
    )
    gpu_info = result.stdout.strip()
    print(f"GPU            : {gpu_info}")
except Exception as e:
    print(f"GPU info unavailable: {e}")

# CuPy device info
try:
    import cupy as cp
    dev = cp.cuda.Device(0)
    props = cp.cuda.runtime.getDeviceProperties(dev.id)
    mem_total = props['totalGlobalMem'] / 1e9
    print(f"CuPy device    : {props['name'].decode()} | {mem_total:.1f} GB VRAM")
    print(f"CUDA version   : {cp.cuda.runtime.runtimeGetVersion()}")
    HAVE_GPU = True
except ImportError:
    print("CuPy not found — GPU cells will be skipped.")
    HAVE_GPU = False

print("=" * 60)

## Cell 2 — Imports & Helpers

In [ ]:
# ── Cell 2: Imports & Benchmark Helper ──────────────────────────────────────
# My own code — timing utilities and performance metric calculations

import numpy as np
import scipy.sparse as sp
import time, warnings
from tabulate import tabulate  # pip install tabulate if needed

try:
    from tabulate import tabulate
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "-q", "tabulate"])
    from tabulate import tabulate

warnings.filterwarnings('ignore')

DTYPE = np.float64   # FP64 throughout — required by assignment
WARMUP = 3           # warmup runs (not timed)
NRUNS  = 10          # timed runs — report median

def bench_cpu(fn, *args, warmup=WARMUP, nruns=NRUNS):
    """Benchmark a CPU callable. Returns median wall-clock time in seconds."""
    for _ in range(warmup):
        fn(*args)
    times = []
    for _ in range(nruns):
        t0 = time.perf_counter()
        fn(*args)
        times.append(time.perf_counter() - t0)
    return float(np.median(times))

def bench_gpu(fn, *args, warmup=WARMUP, nruns=NRUNS):
    """Benchmark a GPU callable using CUDA events for accurate timing."""
    import cupy as cp
    for _ in range(warmup):
        fn(*args)
    cp.cuda.Stream.null.synchronize()
    times = []
    for _ in range(nruns):
        start = cp.cuda.Event(); end = cp.cuda.Event()
        start.record()
        fn(*args)
        end.record()
        end.synchronize()
        times.append(cp.cuda.get_elapsed_time(start, end) / 1e3)  # ms -> s
    return float(np.median(times))

def gemv_flops(M, N):
    """FLOPs for y = A*x: 2*M*N (one multiply + one add per element)."""
    return 2 * M * N

def gemv_bytes(M, N, dtype=DTYPE):
    """Bytes read/written for GEMV: A (MxN) + x (N) + y (M), all FP64."""
    itemsize = np.dtype(dtype).itemsize  # 8 bytes for float64
    return (M * N + N + M) * itemsize

def spmv_flops(nnz):
    """FLOPs for SpMV: 2 * nnz (one multiply + one add per nonzero)."""
    return 2 * nnz

def spmv_bytes_csr(M, nnz, dtype=DTYPE):
    """Bytes for CSR SpMV: values + col_indices + row_ptr + x + y."""
    itemsize = np.dtype(dtype).itemsize
    val_bytes  = nnz * itemsize          # float64 values
    col_bytes  = nnz * 4                 # int32 col indices
    ptr_bytes  = (M + 1) * 4            # int32 row pointers
    x_bytes    = (nnz // M) * itemsize  # approx x elements touched
    y_bytes    = M * itemsize           # output vector
    return val_bytes + col_bytes + ptr_bytes + x_bytes + y_bytes

def report(label, t_sec, flops, nbytes):
    """Return a dict row for the results table."""
    return {
        'Variant'  : label,
        'Time (ms)': f"{t_sec*1e3:.3f}",
        'GFLOP/s'  : f"{flops/t_sec/1e9:.3f}",
        'GB/s'     : f"{nbytes/t_sec/1e9:.3f}"
    }

results = []  # accumulate all rows
print("Helpers loaded. dtype=", DTYPE, "| warmup=", WARMUP, "| runs=", NRUNS)

## Cell 3 — Matrix Sizes
We test two shapes as required by Track A:
- **Square:** 8192 × 8192 (~512 MB for FP64)
- **Tall-skinny:** 16384 × 512 (more rows than columns — common in ML)

In [ ]:
# ── Cell 3: Matrix Sizes & Data Generation ───────────────────────────────────
# My own code — generates random FP64 test matrices and vectors

SIZES = [
    (8192,  8192,  "Square 8192x8192"),
    (16384, 512,   "Tall-skinny 16384x512"),
]

SPARSITY = 0.01  # 1% nonzeros — typical for sparse problems

rng = np.random.default_rng(42)

def make_dense(M, N):
    A = rng.standard_normal((M, N)).astype(DTYPE)
    x = rng.standard_normal(N).astype(DTYPE)
    return A, x

def make_sparse_csr(M, N, density=SPARSITY):
    """Random sparse matrix in CSR format."""
    A_sp = sp.random(M, N, density=density, format='csr', dtype=DTYPE,
                     random_state=42)
    x = rng.standard_normal(N).astype(DTYPE)
    return A_sp, x

for M, N, label in SIZES:
    mem_mb = M * N * 8 / 1e6
    nnz = int(M * N * SPARSITY)
    print(f"{label:30s} | Dense mem: {mem_mb:7.1f} MB | Sparse nnz: {nnz:,}")

print(f"\nAll matrices: dtype=float64, sparsity={SPARSITY*100:.0f}%")

## Cell 4 — CPU Dense GEMV (NumPy)
**PCAM view:** Rows of A partitioned across threads. Each thread computes a contiguous block of y. No inter-thread communication needed (embarrassingly parallel row-wise).

In [ ]:
# ── Cell 4: CPU Dense GEMV ────────────────────────────────────────────────────
# My own benchmarking code; NumPy calls BLAS dgemv internally

print("=" * 60)
print("CPU DENSE GEMV (NumPy / BLAS dgemv, FP64)")
print("=" * 60)

for M, N, label in SIZES:
    A, x = make_dense(M, N)

    # Correctness check vs explicit loop on small slice
    y_ref = A @ x
    assert y_ref.shape == (M,), "Shape mismatch!"

    # Benchmark
    t = bench_cpu(lambda: A @ x)
    flops = gemv_flops(M, N)
    nbytes = gemv_bytes(M, N)

    row = report(f"CPU GEMV | {label}", t, flops, nbytes)
    results.append(row)
    print(f"  {label}: {t*1e3:.3f} ms | {flops/t/1e9:.3f} GFLOP/s | {nbytes/t/1e9:.3f} GB/s")

print("\nNote: NumPy uses OpenBLAS/MKL internally — parallelism is implicit.")
print("PCAM: Rows of A partitioned (row-striped). Communication: none (shared memory).")

## Cell 5 — CPU Sparse SpMV (SciPy CSR)
**Format choice:** CSR (Compressed Sparse Row) — standard for irregular sparsity patterns. HYB/ELL would help only if row lengths are uniform.

In [ ]:
# ── Cell 5: CPU Sparse SpMV (SciPy CSR) ──────────────────────────────────────
# My own benchmarking code; uses scipy.sparse CSR matrix-vector product

print("=" * 60)
print("CPU SPARSE SpMV (SciPy CSR, FP64)")
print("=" * 60)

for M, N, label in SIZES:
    A_sp, x = make_sparse_csr(M, N, density=SPARSITY)
    nnz = A_sp.nnz

    # Correctness: compare CSR result to dense NumPy on same data
    A_dense = A_sp.toarray()
    y_ref  = A_dense @ x
    y_csr  = A_sp @ x
    assert np.allclose(y_ref, y_csr, rtol=1e-10), "CSR result mismatch!"

    t = bench_cpu(lambda: A_sp @ x)
    flops  = spmv_flops(nnz)
    nbytes = spmv_bytes_csr(M, nnz)

    row = report(f"CPU SpMV CSR | {label}", t, flops, nbytes)
    results.append(row)
    print(f"  {label}: {t*1e3:.3f} ms | {flops/t/1e9:.3f} GFLOP/s | {nbytes/t/1e9:.3f} GB/s")
    print(f"    nnz={nnz:,} | density={SPARSITY*100:.1f}%")

print("\nPCAM: CSR naturally row-partitions. Communication: shared x vector (broadcast).")
print("Format note: CSR chosen over ELL because row lengths vary (random sparsity).")

## Cell 6 — GPU Dense GEMV (CuPy / cuBLAS)
**PCAM view:** Same row-striped partition as CPU, but now each CUDA thread block handles a tile of rows. cuBLAS `dgemv` manages all of this internally.

In [ ]:
# ── Cell 6: GPU Dense GEMV (CuPy / cuBLAS) ───────────────────────────────────
# My own code — uses CuPy which dispatches to cuBLAS dgemv under the hood

if not HAVE_GPU:
    print("No GPU detected — skipping GPU cells.")
else:
    import cupy as cp

    print("=" * 60)
    print("GPU DENSE GEMV (CuPy → cuBLAS dgemv, FP64)")
    print("=" * 60)

    for M, N, label in SIZES:
        A_cpu, x_cpu = make_dense(M, N)

        # Transfer to GPU
        A_gpu = cp.asarray(A_cpu)
        x_gpu = cp.asarray(x_cpu)

        # Correctness: compare GPU result to CPU NumPy
        y_gpu = A_gpu @ x_gpu
        y_cpu = A_cpu @ x_cpu
        assert np.allclose(cp.asnumpy(y_gpu), y_cpu, rtol=1e-8), "GPU GEMV mismatch!"

        t = bench_gpu(lambda: A_gpu @ x_gpu)
        flops  = gemv_flops(M, N)
        nbytes = gemv_bytes(M, N)

        row = report(f"GPU GEMV | {label}", t, flops, nbytes)
        results.append(row)
        print(f"  {label}: {t*1e3:.3f} ms | {flops/t/1e9:.3f} GFLOP/s | {nbytes/t/1e9:.3f} GB/s")

    print("\nNote: cuBLAS dgemv uses tiled, coalesced access patterns internally.")
    print("H2D transfer NOT included in timing — pure kernel throughput reported.")
    print("PCAM: Row-striped partition; threads in a warp handle adjacent rows → coalesced.")

## Cell 7 — GPU Sparse SpMV (CuPy / cuSPARSE)
**Format choice:** CSR with cuSPARSE adaptive algorithm. For our random sparsity (uniform row lengths), ELL/HYB would offer marginal gain — CSR is safer and more general.

In [ ]:
# ── Cell 7: GPU Sparse SpMV (CuPy / cuSPARSE) ────────────────────────────────
# My own code — CuPy sparse uses cuSPARSE internally for CSR SpMV

if not HAVE_GPU:
    print("No GPU detected — skipping GPU cells.")
else:
    import cupy as cp
    import cupyx.scipy.sparse as cpsp

    print("=" * 60)
    print("GPU SPARSE SpMV (CuPy CSR → cuSPARSE, FP64)")
    print("=" * 60)

    for M, N, label in SIZES:
        A_sp, x_cpu = make_sparse_csr(M, N, density=SPARSITY)
        nnz = A_sp.nnz

        # Transfer sparse matrix and vector to GPU
        A_gpu_sp = cpsp.csr_matrix(A_sp)
        x_gpu    = cp.asarray(x_cpu)

        # Correctness check
        y_cpu_ref = A_sp @ x_cpu
        y_gpu_res = A_gpu_sp @ x_gpu
        assert np.allclose(cp.asnumpy(y_gpu_res), y_cpu_ref, rtol=1e-8), "GPU SpMV mismatch!"

        t = bench_gpu(lambda: A_gpu_sp @ x_gpu)
        flops  = spmv_flops(nnz)
        nbytes = spmv_bytes_csr(M, nnz)

        row = report(f"GPU SpMV CSR | {label}", t, flops, nbytes)
        results.append(row)
        print(f"  {label}: {t*1e3:.3f} ms | {flops/t/1e9:.3f} GFLOP/s | {nbytes/t/1e9:.3f} GB/s")
        print(f"    nnz={nnz:,} | density={SPARSITY*100:.1f}%")

    print("\nFormat note: CSR chosen. ELL would help if all rows had equal nnz (they don't here).")
    print("cuSPARSE uses adaptive CSR algorithm: merge-based path for load balancing.")

## Cell 8 — Summary Performance Table

In [ ]:
# ── Cell 8: Summary Table ─────────────────────────────────────────────────────
# My own code — prints formatted results table

print("\n" + "=" * 70)
print("PERFORMANCE SUMMARY (FP64, median of 10 runs, no H2D transfer time)")
print("=" * 70)
print(tabulate(results,
               headers='keys',
               tablefmt='github',
               stralign='left'))

print("\nAnnotations:")
print("  - Dense GEMV is BANDWIDTH-BOUND (low arithmetic intensity ~0.25 FLOP/byte for FP64)")
print("  - Sparse SpMV is even more bandwidth-bound due to irregular memory access")
print("  - GPU speedup over CPU is typically 3-10x for these sizes on a T4")

## Cell 9 — Arithmetic Intensity Calculation
This feeds directly into the Roofline plot.

In [ ]:
# ── Cell 9: Arithmetic Intensity ──────────────────────────────────────────────
# My own analysis code — computes AI for Roofline plot

print("=" * 60)
print("ARITHMETIC INTENSITY (FLOP / byte) — inputs to Roofline")
print("=" * 60)

for M, N, label in SIZES:
    # Dense GEMV
    ai_dense = gemv_flops(M, N) / gemv_bytes(M, N)

    # Sparse SpMV
    nnz = int(M * N * SPARSITY)
    ai_sparse = spmv_flops(nnz) / spmv_bytes_csr(M, nnz)

    print(f"  {label}")
    print(f"    Dense  GEMV AI : {ai_dense:.4f} FLOP/byte")
    print(f"    Sparse SpMV AI : {ai_sparse:.4f} FLOP/byte")
    print()

print("Interpretation:")
print("  AI << 1 means heavily bandwidth-bound (typical for MxV).")
print("  On a T4: peak FP64 = ~250 GFLOP/s, peak BW = ~300 GB/s")
print("  Roofline ridge point = 250/300 ≈ 0.83 FLOP/byte")
print("  Both dense and sparse fall well BELOW the ridge → BW-bound.")

## Cell 10 — Correctness Sanity Checks (vs NumPy ground truth)

In [ ]:
# ── Cell 10: Sanity Checks ────────────────────────────────────────────────────
# My own verification code — ensures all implementations agree

print("=" * 60)
print("CORRECTNESS CHECKS (all vs NumPy ground truth)")
print("=" * 60)

M, N = 1024, 1024
A_np, x_np = make_dense(M, N)
y_ref = A_np @ x_np

# CPU sparse
A_sp, _ = make_sparse_csr(M, N, density=0.1)
A_sp_dense = A_sp.toarray()
x_sp = x_np
y_sp_ref = A_sp_dense @ x_sp
y_sp_csr = A_sp @ x_sp
print(f"  CPU CSR SpMV vs NumPy dense: max diff = {np.max(np.abs(y_sp_ref - y_sp_csr)):.2e} ✓")

if HAVE_GPU:
    import cupy as cp
    import cupyx.scipy.sparse as cpsp

    A_gpu = cp.asarray(A_np)
    x_gpu = cp.asarray(x_np)
    y_gpu = cp.asnumpy(A_gpu @ x_gpu)
    print(f"  GPU GEMV vs NumPy dense   : max diff = {np.max(np.abs(y_ref - y_gpu)):.2e} ✓")

    A_sp_gpu = cpsp.csr_matrix(A_sp)
    y_sp_gpu = cp.asnumpy(A_sp_gpu @ cp.asarray(x_sp))
    print(f"  GPU CSR SpMV vs NumPy dense: max diff = {np.max(np.abs(y_sp_ref - y_sp_gpu)):.2e} ✓")

print("\nAll results within FP64 tolerance. ✓")

## Cell 11 — PCAM Design Summary
Required for Track A — documents your decomposition decisions.

In [ ]:
# ── Cell 11: PCAM Summary (markdown display) ──────────────────────────────────
# My own analysis — documents PCAM design choices for Track A

pcam = """
PCAM DESIGN SUMMARY — y = Ax (GEMV / SpMV)
============================================

P — PARTITIONING
  Dense GEMV : Row-wise partition of A. Each task computes one row of y = dot(A[i,:], x).
               → M primitive tasks (one per output element).
  Sparse SpMV: Same row-wise partition of CSR rows. Tasks vary in cost (different nnz/row).
               → Load imbalance risk with naive partition.

C — COMMUNICATION
  Shared memory (CPU/GPU): x is read-only and shared. No explicit communication.
  Distributed (MPI):       x must be broadcast to all ranks (MPI_Bcast).
                           y partial results must be gathered (MPI_Gather / MPI_Allreduce).
  GPU:                     x fits in L2 cache for medium N; repeated reads are cached.

A — AGGLOMERATION
  CPU  : OpenMP/BLAS groups rows into chunks per thread. Chunk size ≈ M/nthreads.
  GPU  : CUDA blocks handle groups of rows. 1 warp per row is common (cuBLAS strategy).
  MPI  : Each rank holds a contiguous block of rows (row-striped decomposition).

M — MAPPING
  CPU  : OS maps OpenMP threads to physical cores. NUMA-aware allocation helps.
  GPU  : SM scheduler assigns warps to SMs. cuBLAS auto-tunes block dimensions.
  MPI  : Round-robin rank-to-node mapping is default; locality optimizations possible.

BOTTLENECK ANALYSIS
  Dense GEMV AI ≈ 0.25 FLOP/byte → below ridge point → BANDWIDTH BOUND.
  Sparse SpMV AI ≈ 0.08-0.15 FLOP/byte → even more BANDWIDTH BOUND.
  Key insight: adding more compute units doesn't help here; need faster memory.
"""
print(pcam)

## Cell 12 — Optimization: Tiled Shared Memory GEMV (Numba CUDA)
**Optimization move #1:** Tile x into shared memory to reduce global memory reads.

In [ ]:
# ── Cell 12: Tiled GEMV Kernel (Numba CUDA) ───────────────────────────────────
# My own custom kernel — implements shared-memory tiling optimization
# Optimization: tiles of x loaded into shared memory to reduce global mem pressure

if not HAVE_GPU:
    print("No GPU — skipping custom kernel.")
else:
    try:
        from numba import cuda
        import numpy as np

        TILE = 32  # tile size (must be <= 1024, ideally warp multiple)

        @cuda.jit
        def tiled_gemv_kernel(A, x, y, M, N):
            """
            Tiled GEMV: each thread computes one element of y.
            x is loaded in TILE-sized chunks into shared memory.
            """
            row = cuda.grid(1)  # global thread index = row index
            x_shared = cuda.shared.array(shape=TILE, dtype=np.float64)

            if row < M:
                acc = 0.0
                for tile_start in range(0, N, TILE):
                    # Cooperative load: each thread loads one element of x
                    local_idx = cuda.threadIdx.x
                    global_idx = tile_start + local_idx
                    if global_idx < N:
                        x_shared[local_idx] = x[global_idx]
                    cuda.syncthreads()

                    # Accumulate dot product for this tile
                    for k in range(min(TILE, N - tile_start)):
                        acc += A[row, tile_start + k] * x_shared[k]
                    cuda.syncthreads()

                y[row] = acc

        # Test on square matrix
        M, N = 4096, 4096
        A_np, x_np = make_dense(M, N)
        y_ref = A_np @ x_np

        A_d = cuda.to_device(A_np)
        x_d = cuda.to_device(x_np)
        y_d = cuda.device_array(M, dtype=np.float64)

        threads_per_block = TILE
        blocks = (M + TILE - 1) // TILE

        # Warmup
        for _ in range(3):
            tiled_gemv_kernel[blocks, threads_per_block](A_d, x_d, y_d, M, N)
        cuda.synchronize()

        # Time it
        import time
        times_tiled = []
        for _ in range(10):
            t0 = time.perf_counter()
            tiled_gemv_kernel[blocks, threads_per_block](A_d, x_d, y_d, M, N)
            cuda.synchronize()
            times_tiled.append(time.perf_counter() - t0)

        t = float(np.median(times_tiled))
        y_out = y_d.copy_to_host()

        # Correctness
        max_diff = np.max(np.abs(y_ref - y_out))
        print(f"Tiled GEMV correctness: max diff vs NumPy = {max_diff:.2e}")

        flops = gemv_flops(M, N)
        nbytes = gemv_bytes(M, N)
        row_res = report(f"GPU Tiled GEMV (Numba) | Square 4096x4096", t, flops, nbytes)
        results.append(row_res)
        print(f"  4096x4096: {t*1e3:.3f} ms | {flops/t/1e9:.3f} GFLOP/s | {nbytes/t/1e9:.3f} GB/s")
        print("  Optimization: shared-memory tiling of x reduces redundant global loads.")

    except Exception as e:
        print(f"Numba kernel skipped: {e}")
        print("(This is OK — cuBLAS in Cell 6 is the primary GPU result.)")

## Cell 13 — Final Results Table (all variants)

In [ ]:
# ── Cell 13: Final Summary ────────────────────────────────────────────────────
# My own code

print("\n" + "=" * 70)
print("FINAL PERFORMANCE TABLE — ALL VARIANTS")
print("=" * 70)
print(tabulate(results, headers='keys', tablefmt='github', stralign='left'))

print("""
KEY OBSERVATIONS:
1. Dense GEMV is memory-bandwidth limited (AI ≈ 0.25 FLOP/byte on both CPU and GPU).
2. Sparse SpMV shows lower GFLOP/s than dense — irregular access hurts cache utilization.
3. GPU speedup over CPU is mainly from higher memory bandwidth (T4: ~300 GB/s vs CPU ~50 GB/s).
4. For sparse, CSR is appropriate here; ELL/HYB would only help with uniform row lengths.
5. Both variants fall LEFT of the Roofline ridge → bandwidth bound, not compute bound.
""")